In [3]:
from __future__ import annotations
import math
import numpy as np
from dataclasses import dataclass, field
from typing import List, Tuple

EPS = 1e-12

def safe_log(x: float) -> float:
    return math.log(max(x, EPS))

def sigmoid(x: float) -> float:
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    z = math.exp(x)
    return z / (1.0 + z)

def clip(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, x))

def safe_exp(x: float, lo: float = -35.0, hi: float = 35.0) -> float:
    return math.exp(clip(x, lo, hi))

def gpd_rvs(rng: np.random.Generator, xi: float, sigma: float) -> float:
    u = float(rng.random())
    u = clip(u, EPS, 1.0 - EPS)
    if abs(xi) < 1e-12:
        return -sigma * math.log(1.0 - u)
    return (sigma / xi) * ((1.0 - u) ** (-xi) - 1.0)

def shannon_entropy_signs(signs: np.ndarray) -> float:

    if len(signs) == 0:
        return 0.0
    p = float(np.mean(signs == 1))
    q = 1.0 - p
    def h(a): return 0.0 if a <= 0 else -a * math.log(a)
    return float((h(p) + h(q)) / math.log(2.0))

def max_drawdown_pct(equity: np.ndarray) -> float:

    peak = float(equity[0])
    mdd = 0.0
    for x in equity:
        peak = max(peak, float(x))
        dd = (peak - float(x)) / max(peak, EPS)
        mdd = max(mdd, dd)
    return float(mdd * 100.0)

def hurst_rs(prices: np.ndarray) -> float:

    if len(prices) < 200:
        return 0.5
    x = np.diff(np.log(np.maximum(prices, EPS)))
    n = len(x)
    m = 10
    chunk = n // m
    if chunk < 20:
        return 0.5
    RS = []
    for i in range(m):
        seg = x[i*chunk:(i+1)*chunk]
        seg = seg - seg.mean()
        y = np.cumsum(seg)
        R = y.max() - y.min()
        S = seg.std()
        if S > EPS:
            RS.append(R / S)
    if not RS:
        return 0.5
    H = math.log(float(np.mean(RS))) / math.log(chunk)
    return float(clip(H, 0.0, 1.0))

def quantum_repsi(returns: np.ndarray, rng: np.random.Generator) -> float:

    if len(returns) < 50:
        return 0.0
    r = returns[-200:]
    thr = max(float(r.std()) * 0.25, 1e-8)
    up = float(np.mean(r > thr))
    down = float(np.mean(r < -thr))
    flat = max(0.0, 1.0 - up - down)
    p = np.array([down, flat, up], dtype=float)
    p = p / max(float(p.sum()), EPS)
    k = int(np.argmax(p))
    phi = float(rng.uniform(0.0, 2.0 * math.pi))
    return float(math.sqrt(p[k]) * math.cos(phi))

def es_left_tail(pnl_pct: np.ndarray, alpha: float = 0.99) -> float:

    if len(pnl_pct) == 0:
        return 0.0
    q = float(np.quantile(pnl_pct, 1.0 - alpha))
    tail = pnl_pct[pnl_pct <= q]
    if len(tail) == 0:
        return 0.0
    return float(-tail.mean())

def losses_clipped(pnl_pct: np.ndarray) -> np.ndarray:
    return np.maximum(0.0, -pnl_pct)

@dataclass
class MarketParams:
    tick: float = 1e-4
    base_mid: float = 0.02
    levels: int = 20
    base_depth: float = 2e7
    depth_decay: float = 0.25

    base_spread_ticks: int = 2
    spread_vol_sens: float = 140.0
    depth_vol_sens: float = 10.0

    vol_mean: float = 1.5e-4
    vol_kappa: float = 0.04
    vol_of_vol: float = 0.20

    jump_prob: float = 5e-4
    jump_scale: float = 10.0

    dt_seconds: float = 1.0

@dataclass
class MarketState:
    mid: float
    best_bid: float
    best_ask: float
    spread: float
    spread_ticks: int
    depth_exec: float
    imbalance: float
    microprice: float
    vol_step: float
    vol_1m: float
    bid_prices: np.ndarray
    bid_qty: np.ndarray
    ask_prices: np.ndarray
    ask_qty: np.ndarray

@dataclass
class SyntheticMarket:
    p: MarketParams
    rng: np.random.Generator
    mid: float = field(init=False)
    vol: float = field(init=False)
    bids_q: np.ndarray = field(init=False)
    asks_q: np.ndarray = field(init=False)
    spread_ticks: int = field(init=False)
    mid_hist: List[float] = field(default_factory=list, init=False)

    def __post_init__(self):
        self.mid = self.p.base_mid
        self.vol = self.p.vol_mean
        self._init_book()
        self.mid_hist = [self.mid]

    def _depth_profile(self) -> np.ndarray:
        lvl = np.arange(self.p.levels, dtype=float)
        shape = np.exp(-self.p.depth_decay * lvl)
        noise = np.exp(self.rng.normal(0.0, 0.25, size=self.p.levels))
        prof = self.p.base_depth * shape * noise / max(float(noise.mean()), EPS)
        return np.maximum(prof, 1e3)

    def _init_book(self):
        self.spread_ticks = self.p.base_spread_ticks
        self.bids_q = self._depth_profile()
        self.asks_q = self._depth_profile()

    def step(self, scenario: str, t: int) -> MarketState:
        shock_vol_mult = 1.0
        shock_depth_mult = 1.0
        jump_prob = self.p.jump_prob

        if scenario == "covid" and 2000 <= t <= 6000:
            shock_vol_mult = 3.0
            shock_depth_mult = 0.35
            jump_prob *= 5.0
        if scenario == "chf2015" and t == 3000:
            shock_vol_mult = 15.0
            shock_depth_mult = 0.15
            jump_prob *= 100.0
        logv = safe_log(self.vol)
        logv_mean = safe_log(self.p.vol_mean)
        logv = logv + self.p.vol_kappa * (logv_mean - logv) + self.p.vol_of_vol * float(self.rng.normal(0.0, 1.0))
        self.vol = max(EPS, math.exp(logv)) * shock_vol_mult
        dm = float(self.rng.normal(0.0, self.vol))
        if float(self.rng.random()) < jump_prob:
            dm += self.p.jump_scale * self.vol * float(self.rng.normal(0.0, 1.0))
        self.mid = max(self.p.tick, self.mid + dm)
        sp = self.p.base_spread_ticks + int(self.p.spread_vol_sens * (self.vol / max(self.p.vol_mean, EPS)))
        self.spread_ticks = int(clip(sp, self.p.base_spread_ticks, 250))
        depth_target = self.p.base_depth / (1.0 + self.p.depth_vol_sens * (self.vol / max(self.p.vol_mean, EPS)))
        depth_target *= shock_depth_mult
        old_base = self.p.base_depth
        self.p.base_depth = max(5e5, depth_target)
        self.bids_q = 0.7 * self.bids_q + 0.3 * self._depth_profile()
        self.asks_q = 0.7 * self.asks_q + 0.3 * self._depth_profile()
        self.p.base_depth = old_base
        best_bid = self.mid - 0.5 * self.spread_ticks * self.p.tick
        best_ask = self.mid + 0.5 * self.spread_ticks * self.p.tick
        spread = max(best_ask - best_bid, self.p.tick)
        lvl = np.arange(self.p.levels, dtype=float)
        bid_prices = best_bid - lvl * self.p.tick
        ask_prices = best_ask + lvl * self.p.tick
        topN = min(5, self.p.levels)
        depth_exec = float(self.bids_q[:topN].sum() + self.asks_q[:topN].sum())
        depth_exec = max(depth_exec, 1e3)
        imb = float((self.bids_q[:topN].sum() - self.asks_q[:topN].sum()) / depth_exec)
        microprice = float((best_ask * self.bids_q[0] + best_bid * self.asks_q[0]) / max(self.bids_q[0] + self.asks_q[0], EPS))
        vol_1m = self.vol * math.sqrt(60.0 / max(self.p.dt_seconds, EPS))
        self.mid_hist.append(self.mid)
        if len(self.mid_hist) > 2000:
            self.mid_hist = self.mid_hist[-2000:]

        return MarketState(
            mid=self.mid, best_bid=best_bid, best_ask=best_ask, spread=spread,
            spread_ticks=self.spread_ticks, depth_exec=depth_exec, imbalance=imb,
            microprice=microprice, vol_step=self.vol, vol_1m=vol_1m,
            bid_prices=bid_prices, bid_qty=self.bids_q.copy(), ask_prices=ask_prices, ask_qty=self.asks_q.copy()
        )


def ricci_proxy(history: np.ndarray) -> Tuple[float, float]:
    if history.shape[0] < 50:
        return 0.0, 0.0
    X = history[-400:]
    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    sd = np.where(sd < 1e-6, 1.0, sd)
    Z = (X - mu) / sd
    dZ = np.diff(Z, axis=0)
    C = np.cov(dZ.T)
    C = np.nan_to_num(C, nan=0.0, posinf=0.0, neginf=0.0)
    Ric = -0.5 * (C + C.T)
    tr = float(np.trace(Ric))
    fro = float(np.linalg.norm(Ric, ord="fro"))
    tr = float(clip(tr, -5.0, 5.0))
    fro = float(clip(fro, 0.0, 10.0))
    return tr, fro

@dataclass
class RFQ:
    eps: int
    v: float
    tier: int
    hour: int

@dataclass
class RFQParams:
    intensity: float = 0.20
    tier_probs: Tuple[float, float, float] = (0.2, 0.6, 0.2)
    v_logn_mu: float = math.log(2e6)
    v_logn_sig: float = 0.8
    sign_bias: float = 0.0
    v_min: float = 5e4
    v_max: float = 1e7

def sample_rfqs(rng: np.random.Generator, p: RFQParams, hour: int, vol_1m: float) -> List[RFQ]:
    lam = p.intensity * (1.0 + 6.0 * (vol_1m / 0.01))
    n = int(rng.poisson(lam))
    if n <= 0:
        return []
    tiers = rng.choice([0, 1, 2], size=n, p=np.array(p.tier_probs))
    v = rng.lognormal(mean=p.v_logn_mu, sigma=p.v_logn_sig, size=n)
    v = np.clip(v, p.v_min, p.v_max)
    prob_buy = sigmoid(p.sign_bias + 2.0 * (vol_1m - 0.01))
    eps = np.where(rng.random(n) < prob_buy, 1, -1)
    return [RFQ(int(eps[i]), float(v[i]), int(tiers[i]), int(hour)) for i in range(n)]


@dataclass
class QCSRFParams:
    alpha: float = 0.0
    beta: float = -0.25
    gamma_X: np.ndarray = field(default_factory=lambda: np.array([0.35, 0.15, 0.30, 0.10]))
    phi_E: float = 0.7
    phi_H: float = 0.6
    phi_S: float = 0.4
    theta: float = 0.3

    eta: float = 1.5e-6
    xi: float = 0.35
    sigma_jump: float = 2.0
    lambda0_slip: float = 0.10
    Kmin: float = 0.60
    lambda_slip_max: float = 8.0
    jump_bp_cap: float = 100.0

    lambda0_inv: float = 0.10
    kE: float = 0.8
    kH: float = 1.2
    delta: float = 0.8
    lambda_inv_max: float = 5.0

    I_safe: float = 6e6
    I_max: float = 2e7
    forced_hedge_k: float = 2e-6

    s_min: float = 0.5
    s_max: float = 40.0

    k0: float = 0.05
    gamma_entropy: float = 2.0
    hedge_eps: float = 1e-8

@dataclass
class QCSRFState:
    I: float = 0.0
    k_hedge: float = 0.0
    last_H: float = 0.5

class QCSRF:
    def __init__(self, p: QCSRFParams, rng: np.random.Generator):
        self.p = p
        self.rng = rng

    def p_hit(self, s_bp: float, X: np.ndarray, E: float, Eref: float, H: float, S: float, repsi: float) -> float:
        u = self.p.alpha + self.p.beta * s_bp + float(self.p.gamma_X @ X)
        u += self.p.phi_E * ((E - Eref) / max(Eref, EPS))
        u += self.p.phi_H * (H - 0.5)
        u += -self.p.phi_S * S
        u += self.p.theta * repsi
        return clip(sigmoid(u), 0.0, 1.0)

    def lambda_slip(self, froRic: float) -> float:
        lam = self.p.lambda0_slip * safe_exp(-froRic / max(self.p.Kmin, EPS))
        return float(clip(lam, 0.0, self.p.lambda_slip_max))

    def lambda_inv(self, E: float, Eref: float, H: float, trRic: float) -> float:
        base = self.p.lambda0_inv * (1.0 + self.p.kE * (E / max(Eref, EPS)) + self.p.kH * (H - 0.5) ** 2)
        lam = base * safe_exp(self.p.delta * trRic, lo=-20, hi=20)
        return float(clip(lam, 0.0, self.p.lambda_inv_max))

    def c_slip_bp(self, v: float, depth: float, lam_slip: float) -> float:
        base = self.p.eta * (v / max(depth, EPS))
        dN = int(self.rng.poisson(lam_slip))
        jump = 0.0
        if dN > 0:
            jump = gpd_rvs(self.rng, self.p.xi, self.p.sigma_jump) * float(dN)
        return float(clip(base + jump, 0.0, self.p.jump_bp_cap))

    def cost_inv(self, I: float, eps: int, v: float, lam_inv: float) -> float:
        I_m = I / 1e6
        v_m = v / 1e6
        return float((lam_inv / 2.0) * (((I_m + eps * v_m) ** 2) - (I_m ** 2)))

    def solve_s_star(self, v: float, X: np.ndarray, E: float, Eref: float, H: float, S: float, repsi: float, c_slip_bp: float) -> float:
        lo, hi = self.p.s_min, self.p.s_max
        c_over_v = c_slip_bp / max(v, EPS)

        def f(s):
            p = self.p_hit(s, X, E, Eref, H, S, repsi)
            return 1.0 - self.p.beta * (s - c_over_v) * (1.0 - p)

        flo, fhi = f(lo), f(hi)
        if flo * fhi > 0:
            grid = np.linspace(lo, hi, 140)
            best_s, best_J = lo, -1e18
            for s in grid:
                p = self.p_hit(float(s), X, E, Eref, H, S, repsi)
                J = p * (s - c_over_v)
                if J > best_J:
                    best_J, best_s = float(J), float(s)
            return float(best_s)

        for _ in range(80):
            mid = 0.5 * (lo + hi)
            fm = f(mid)
            if abs(fm) < 1e-6:
                return float(mid)
            if flo * fm <= 0:
                hi = mid
                fhi = fm
            else:
                lo = mid
                flo = fm
        return float(0.5 * (lo + hi))

@dataclass
class BacktestOut:
    equity: np.ndarray
    pnl: np.ndarray
    trades: int
    rfqs: int
    fill_rate: float
    net_pnl_pct: float
    max_dd_pct: float
    es_tick_099_pct: float
    es_day_099_pct: float
    final_inventory: float

def run_backtest(
    steps: int = 3000,
    scenario: str = "normal",
    seed: int = 42,
    ticks_per_day: int = 1000,
    inv_scale: float = 250.0
) -> BacktestOut:

    rng = np.random.default_rng(seed)
    market = SyntheticMarket(MarketParams(), rng)
    rfqp = RFQParams()
    qpar = QCSRFParams()
    model = QCSRF(qpar, rng)
    st = QCSRFState()
    equity0 = 1_000_000.0
    eq = equity0
    equity = np.zeros(steps, dtype=float)
    pnl = np.zeros(steps, dtype=float)
    hour = 8
    state_hist: List[np.ndarray] = []
    sign_hist: List[int] = []
    trades = 0
    rfq_count = 0
    Eref = 0.55

    for t in range(steps):
        if t % 400 == 0:
            hour = (hour + 1) % 24
        mk = market.step(scenario, t)
        x = np.array([safe_log(mk.mid), mk.spread, mk.imbalance, safe_log(mk.depth_exec)], dtype=float)
        state_hist.append(x)
        if len(state_hist) > 1200:
            state_hist = state_hist[-1200:]
        Hx = np.vstack(state_hist)
        trRic, froRic = ricci_proxy(Hx)
        mids = np.array(market.mid_hist, dtype=float)
        rets = np.diff(np.log(np.maximum(mids, EPS)))
        repsi = quantum_repsi(rets, rng)
        E = hurst_rs(mids[-800:])
        rfqs = sample_rfqs(rng, rfqp, hour, mk.vol_1m)
        rfq_count += len(rfqs)
        step_pnl = 0.0

        for F in rfqs:
            sign_hist.append(F.eps)
            if len(sign_hist) > 200:
                sign_hist = sign_hist[-200:]
            H = shannon_entropy_signs(np.array(sign_hist, dtype=int))
            S = abs(st.I) / max(qpar.I_safe, EPS)
            X = np.array([mk.vol_1m / 0.01, safe_log(mk.depth_exec) / 20.0, float(F.tier - 1), float(F.hour) / 23.0], dtype=float)
            lam_slip = model.lambda_slip(froRic)
            cslip_bp = model.c_slip_bp(F.v, mk.depth_exec, lam_slip)
            s_star = model.solve_s_star(F.v, X, E, Eref, H, S, repsi, cslip_bp)
            p_hit = model.p_hit(s_star, X, E, Eref, H, S, repsi)

            if float(rng.random()) < p_hit:
                trades += 1
                step_pnl += (s_star * 1e-4) * F.v
                step_pnl -= (cslip_bp * 1e-4) * F.v
                st.I += F.eps * F.v
                st.I = clip(st.I, -3.0 * qpar.I_max, 3.0 * qpar.I_max)
                lam_inv = model.lambda_inv(E, Eref, H, trRic)
                cinv = model.cost_inv(st.I, F.eps, F.v, lam_inv)
                step_pnl -= cinv * inv_scale

                if abs(st.I) > qpar.I_max:
                    step_pnl -= qpar.forced_hedge_k * abs(st.I)
                    st.I = 0.0

            dH = (H - st.last_H)
            k_skew = qpar.k0 * safe_exp(-qpar.gamma_entropy * dH, lo=-10, hi=10)
            z = abs(st.I) / max(qpar.I_safe, EPS)
            z = min(z, 1e4)
            st.k_hedge += qpar.hedge_eps * (z * z - 1.0) * (1.0 if st.I >= 0 else -1.0)
            st.k_hedge = clip(st.k_hedge, -1e6, 1e6)
            u = F.eps * F.v * p_hit
            h = max(0.0, abs(st.I) - qpar.I_safe) * (1.0 if st.I >= 0 else -1.0)
            st.I += (-k_skew * (st.I + u) - st.k_hedge * h) * 0.01
            st.I = clip(st.I, -3.0 * qpar.I_max, 3.0 * qpar.I_max)

            st.last_H = H

        eq += step_pnl
        if not np.isfinite(eq):
            eq = equity0

        pnl[t] = step_pnl
        equity[t] = eq

    fill_rate = trades / max(rfq_count, 1)
    net_pnl_pct = (equity[-1] - equity0) / equity0 * 100.0
    max_dd_pct = max_drawdown_pct(equity)
    pnl_tick_pct = (pnl / equity0) * 100.0
    es_tick_099 = es_left_tail(pnl_tick_pct, 0.99)
    b = max(int(ticks_per_day), 1)
    n = (len(pnl) // b) * b
    if n >= b:
        pnl_day = pnl[:n].reshape(-1, b).sum(axis=1)
        pnl_day_pct = (pnl_day / equity0) * 100.0
        es_day_099 = es_left_tail(pnl_day_pct, 0.99)
    else:
        es_day_099 = float("nan")

    return BacktestOut(
        equity=equity,
        pnl=pnl,
        trades=trades,
        rfqs=rfq_count,
        fill_rate=float(fill_rate),
        net_pnl_pct=float(net_pnl_pct),
        max_dd_pct=float(max_dd_pct),
        es_tick_099_pct=float(es_tick_099),
        es_day_099_pct=float(es_day_099),
        final_inventory=float(st.I),
    )

if __name__ == "__main__":
    out = run_backtest(steps=3000, scenario="normal", seed=42, ticks_per_day=1000, inv_scale=250.0)
    print("=== Scenario: normal ===")
    print("Net PnL %          :", round(out.net_pnl_pct, 4))
    print("Max Drawdown %     :", round(out.max_dd_pct, 4))
    print("ES/CVaR0.99 tick % :", round(out.es_tick_099_pct, 6))
    print("ES/CVaR0.99 day  % :", round(out.es_day_099_pct, 6))
    print("Trades             :", out.trades)
    print("RFQs               :", out.rfqs)
    print("Fill rate          :", round(out.fill_rate, 4))
    print("Final Inventory    :", round(out.final_inventory, 2))


=== Scenario: normal ===
Net PnL %          : 26.113
Max Drawdown %     : 0.4234
ES/CVaR0.99 tick % : 0.000471
ES/CVaR0.99 day  % : -6.861248
Trades             : 289
RFQs               : 1231
Fill rate          : 0.2348
Final Inventory    : -6437365.34


In [4]:
from __future__ import annotations
import math
import numpy as np
from dataclasses import dataclass, field
from typing import List, Tuple

EPS = 1e-12

def safe_log(x: float) -> float:
    return math.log(max(x, EPS))

def sigmoid(x: float) -> float:
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    z = math.exp(x)
    return z / (1.0 + z)

def clip(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, x))

def safe_exp(x: float, lo: float = -35.0, hi: float = 35.0) -> float:
    return math.exp(clip(x, lo, hi))

def gpd_rvs(rng: np.random.Generator, xi: float, sigma: float) -> float:
    u = float(rng.random())
    u = clip(u, EPS, 1.0 - EPS)
    if abs(xi) < 1e-12:
        return -sigma * math.log(1.0 - u)
    return (sigma / xi) * ((1.0 - u) ** (-xi) - 1.0)

def shannon_entropy_signs(signs: np.ndarray) -> float:
    if len(signs) == 0:
        return 0.0
    p = float(np.mean(signs == 1))
    q = 1.0 - p
    def h(a): return 0.0 if a <= 0 else -a * math.log(a)
    return float((h(p) + h(q)) / math.log(2.0))

def max_drawdown_pct(equity: np.ndarray) -> float:
    peak = float(equity[0])
    mdd = 0.0
    for x in equity:
        peak = max(peak, float(x))
        dd = (peak - float(x)) / max(peak, EPS)
        mdd = max(mdd, dd)
    return float(mdd * 100.0)

def hurst_rs(prices: np.ndarray) -> float:
    if len(prices) < 200:
        return 0.5
    x = np.diff(np.log(np.maximum(prices, EPS)))
    n = len(x)
    m = 10
    chunk = n // m
    if chunk < 20:
        return 0.5
    RS = []
    for i in range(m):
        seg = x[i*chunk:(i+1)*chunk]
        seg = seg - seg.mean()
        y = np.cumsum(seg)
        R = y.max() - y.min()
        S = seg.std()
        if S > EPS:
            RS.append(R / S)
    if not RS:
        return 0.5
    H = math.log(float(np.mean(RS))) / math.log(chunk)
    return float(clip(H, 0.0, 1.0))

def quantum_repsi(returns: np.ndarray, rng: np.random.Generator) -> float:
    if len(returns) < 50:
        return 0.0
    r = returns[-200:]
    thr = max(float(r.std()) * 0.25, 1e-8)
    up = float(np.mean(r > thr))
    down = float(np.mean(r < -thr))
    flat = max(0.0, 1.0 - up - down)
    p = np.array([down, flat, up], dtype=float)
    p = p / max(float(p.sum()), EPS)
    k = int(np.argmax(p))
    phi = float(rng.uniform(0.0, 2.0 * math.pi))
    return float(math.sqrt(p[k]) * math.cos(phi))

# =========================
# PATCH ICI (ES >= 0)
# =========================
def es_left_tail(pnl_pct: np.ndarray, alpha: float = 0.99) -> float:
    """
    ES/CVaR "loss-style" sur la queue gauche du PnL%.
    Retourne une PERTE positive (>=0).
    Si les pires réalisations sont positives -> ES = 0.
    """
    if len(pnl_pct) == 0:
        return 0.0
    q = float(np.quantile(pnl_pct, 1.0 - alpha))     # quantile 1%
    tail = pnl_pct[pnl_pct <= q]
    if len(tail) == 0:
        return 0.0
    tail_losses = np.maximum(0.0, -tail)             # pertes positives seulement
    return float(tail_losses.mean())

def losses_clipped(pnl_pct: np.ndarray) -> np.ndarray:
    return np.maximum(0.0, -pnl_pct)

@dataclass
class MarketParams:
    tick: float = 1e-4
    base_mid: float = 0.02
    levels: int = 20
    base_depth: float = 2e7
    depth_decay: float = 0.25

    base_spread_ticks: int = 2
    spread_vol_sens: float = 140.0
    depth_vol_sens: float = 10.0

    vol_mean: float = 1.5e-4
    vol_kappa: float = 0.04
    vol_of_vol: float = 0.20

    jump_prob: float = 5e-4
    jump_scale: float = 10.0

    dt_seconds: float = 1.0

@dataclass
class MarketState:
    mid: float
    best_bid: float
    best_ask: float
    spread: float
    spread_ticks: int
    depth_exec: float
    imbalance: float
    microprice: float
    vol_step: float
    vol_1m: float
    bid_prices: np.ndarray
    bid_qty: np.ndarray
    ask_prices: np.ndarray
    ask_qty: np.ndarray

@dataclass
class SyntheticMarket:
    p: MarketParams
    rng: np.random.Generator
    mid: float = field(init=False)
    vol: float = field(init=False)
    bids_q: np.ndarray = field(init=False)
    asks_q: np.ndarray = field(init=False)
    spread_ticks: int = field(init=False)
    mid_hist: List[float] = field(default_factory=list, init=False)

    def __post_init__(self):
        self.mid = self.p.base_mid
        self.vol = self.p.vol_mean
        self._init_book()
        self.mid_hist = [self.mid]

    def _depth_profile(self) -> np.ndarray:
        lvl = np.arange(self.p.levels, dtype=float)
        shape = np.exp(-self.p.depth_decay * lvl)
        noise = np.exp(self.rng.normal(0.0, 0.25, size=self.p.levels))
        prof = self.p.base_depth * shape * noise / max(float(noise.mean()), EPS)
        return np.maximum(prof, 1e3)

    def _init_book(self):
        self.spread_ticks = self.p.base_spread_ticks
        self.bids_q = self._depth_profile()
        self.asks_q = self._depth_profile()

    def step(self, scenario: str, t: int) -> MarketState:
        shock_vol_mult = 1.0
        shock_depth_mult = 1.0
        jump_prob = self.p.jump_prob

        if scenario == "covid" and 2000 <= t <= 6000:
            shock_vol_mult = 3.0
            shock_depth_mult = 0.35
            jump_prob *= 5.0
        if scenario == "chf2015" and t == 3000:
            shock_vol_mult = 15.0
            shock_depth_mult = 0.15
            jump_prob *= 100.0

        logv = safe_log(self.vol)
        logv_mean = safe_log(self.p.vol_mean)
        logv = logv + self.p.vol_kappa * (logv_mean - logv) + self.p.vol_of_vol * float(self.rng.normal(0.0, 1.0))
        self.vol = max(EPS, math.exp(logv)) * shock_vol_mult

        dm = float(self.rng.normal(0.0, self.vol))
        if float(self.rng.random()) < jump_prob:
            dm += self.p.jump_scale * self.vol * float(self.rng.normal(0.0, 1.0))
        self.mid = max(self.p.tick, self.mid + dm)

        sp = self.p.base_spread_ticks + int(self.p.spread_vol_sens * (self.vol / max(self.p.vol_mean, EPS)))
        self.spread_ticks = int(clip(sp, self.p.base_spread_ticks, 250))

        depth_target = self.p.base_depth / (1.0 + self.p.depth_vol_sens * (self.vol / max(self.p.vol_mean, EPS)))
        depth_target *= shock_depth_mult
        old_base = self.p.base_depth
        self.p.base_depth = max(5e5, depth_target)
        self.bids_q = 0.7 * self.bids_q + 0.3 * self._depth_profile()
        self.asks_q = 0.7 * self.asks_q + 0.3 * self._depth_profile()
        self.p.base_depth = old_base

        best_bid = self.mid - 0.5 * self.spread_ticks * self.p.tick
        best_ask = self.mid + 0.5 * self.spread_ticks * self.p.tick
        spread = max(best_ask - best_bid, self.p.tick)

        lvl = np.arange(self.p.levels, dtype=float)
        bid_prices = best_bid - lvl * self.p.tick
        ask_prices = best_ask + lvl * self.p.tick

        topN = min(5, self.p.levels)
        depth_exec = float(self.bids_q[:topN].sum() + self.asks_q[:topN].sum())
        depth_exec = max(depth_exec, 1e3)

        imb = float((self.bids_q[:topN].sum() - self.asks_q[:topN].sum()) / depth_exec)
        microprice = float((best_ask * self.bids_q[0] + best_bid * self.asks_q[0]) / max(self.bids_q[0] + self.asks_q[0], EPS))
        vol_1m = self.vol * math.sqrt(60.0 / max(self.p.dt_seconds, EPS))

        self.mid_hist.append(self.mid)
        if len(self.mid_hist) > 2000:
            self.mid_hist = self.mid_hist[-2000:]

        return MarketState(
            mid=self.mid, best_bid=best_bid, best_ask=best_ask, spread=spread,
            spread_ticks=self.spread_ticks, depth_exec=depth_exec, imbalance=imb,
            microprice=microprice, vol_step=self.vol, vol_1m=vol_1m,
            bid_prices=bid_prices, bid_qty=self.bids_q.copy(), ask_prices=ask_prices, ask_qty=self.asks_q.copy()
        )

def ricci_proxy(history: np.ndarray) -> Tuple[float, float]:
    if history.shape[0] < 50:
        return 0.0, 0.0
    X = history[-400:]
    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    sd = np.where(sd < 1e-6, 1.0, sd)
    Z = (X - mu) / sd
    dZ = np.diff(Z, axis=0)
    C = np.cov(dZ.T)
    C = np.nan_to_num(C, nan=0.0, posinf=0.0, neginf=0.0)
    Ric = -0.5 * (C + C.T)
    tr = float(np.trace(Ric))
    fro = float(np.linalg.norm(Ric, ord="fro"))
    tr = float(clip(tr, -5.0, 5.0))
    fro = float(clip(fro, 0.0, 10.0))
    return tr, fro

@dataclass
class RFQ:
    eps: int
    v: float
    tier: int
    hour: int

@dataclass
class RFQParams:
    intensity: float = 0.20
    tier_probs: Tuple[float, float, float] = (0.2, 0.6, 0.2)
    v_logn_mu: float = math.log(2e6)
    v_logn_sig: float = 0.8
    sign_bias: float = 0.0
    v_min: float = 5e4
    v_max: float = 1e7

def sample_rfqs(rng: np.random.Generator, p: RFQParams, hour: int, vol_1m: float) -> List[RFQ]:
    lam = p.intensity * (1.0 + 6.0 * (vol_1m / 0.01))
    n = int(rng.poisson(lam))
    if n <= 0:
        return []
    tiers = rng.choice([0, 1, 2], size=n, p=np.array(p.tier_probs))
    v = rng.lognormal(mean=p.v_logn_mu, sigma=p.v_logn_sig, size=n)
    v = np.clip(v, p.v_min, p.v_max)
    prob_buy = sigmoid(p.sign_bias + 2.0 * (vol_1m - 0.01))
    eps = np.where(rng.random(n) < prob_buy, 1, -1)
    return [RFQ(int(eps[i]), float(v[i]), int(tiers[i]), int(hour)) for i in range(n)]

@dataclass
class QCSRFParams:
    alpha: float = 0.0
    beta: float = -0.25
    gamma_X: np.ndarray = field(default_factory=lambda: np.array([0.35, 0.15, 0.30, 0.10]))
    phi_E: float = 0.7
    phi_H: float = 0.6
    phi_S: float = 0.4
    theta: float = 0.3

    eta: float = 1.5e-6
    xi: float = 0.35
    sigma_jump: float = 2.0
    lambda0_slip: float = 0.10
    Kmin: float = 0.60
    lambda_slip_max: float = 8.0
    jump_bp_cap: float = 100.0

    lambda0_inv: float = 0.10
    kE: float = 0.8
    kH: float = 1.2
    delta: float = 0.8
    lambda_inv_max: float = 5.0

    I_safe: float = 6e6
    I_max: float = 2e7
    forced_hedge_k: float = 2e-6

    s_min: float = 0.5
    s_max: float = 40.0

    k0: float = 0.05
    gamma_entropy: float = 2.0
    hedge_eps: float = 1e-8

@dataclass
class QCSRFState:
    I: float = 0.0
    k_hedge: float = 0.0
    last_H: float = 0.5

class QCSRF:
    def __init__(self, p: QCSRFParams, rng: np.random.Generator):
        self.p = p
        self.rng = rng

    def p_hit(self, s_bp: float, X: np.ndarray, E: float, Eref: float, H: float, S: float, repsi: float) -> float:
        u = self.p.alpha + self.p.beta * s_bp + float(self.p.gamma_X @ X)
        u += self.p.phi_E * ((E - Eref) / max(Eref, EPS))
        u += self.p.phi_H * (H - 0.5)
        u += -self.p.phi_S * S
        u += self.p.theta * repsi
        return clip(sigmoid(u), 0.0, 1.0)

    def lambda_slip(self, froRic: float) -> float:
        lam = self.p.lambda0_slip * safe_exp(-froRic / max(self.p.Kmin, EPS))
        return float(clip(lam, 0.0, self.p.lambda_slip_max))

    def lambda_inv(self, E: float, Eref: float, H: float, trRic: float) -> float:
        base = self.p.lambda0_inv * (1.0 + self.p.kE * (E / max(Eref, EPS)) + self.p.kH * (H - 0.5) ** 2)
        lam = base * safe_exp(self.p.delta * trRic, lo=-20, hi=20)
        return float(clip(lam, 0.0, self.p.lambda_inv_max))

    def c_slip_bp(self, v: float, depth: float, lam_slip: float) -> float:
        base = self.p.eta * (v / max(depth, EPS))
        dN = int(self.rng.poisson(lam_slip))
        jump = 0.0
        if dN > 0:
            jump = gpd_rvs(self.rng, self.p.xi, self.p.sigma_jump) * float(dN)
        return float(clip(base + jump, 0.0, self.p.jump_bp_cap))

    def cost_inv(self, I: float, eps: int, v: float, lam_inv: float) -> float:
        I_m = I / 1e6
        v_m = v / 1e6
        return float((lam_inv / 2.0) * (((I_m + eps * v_m) ** 2) - (I_m ** 2)))

    def solve_s_star(self, v: float, X: np.ndarray, E: float, Eref: float, H: float, S: float, repsi: float, c_slip_bp: float) -> float:
        lo, hi = self.p.s_min, self.p.s_max
        c_over_v = c_slip_bp / max(v, EPS)

        def f(s):
            p = self.p_hit(s, X, E, Eref, H, S, repsi)
            return 1.0 - self.p.beta * (s - c_over_v) * (1.0 - p)

        flo, fhi = f(lo), f(hi)
        if flo * fhi > 0:
            grid = np.linspace(lo, hi, 140)
            best_s, best_J = lo, -1e18
            for s in grid:
                p = self.p_hit(float(s), X, E, Eref, H, S, repsi)
                J = p * (s - c_over_v)
                if J > best_J:
                    best_J, best_s = float(J), float(s)
            return float(best_s)

        for _ in range(80):
            mid = 0.5 * (lo + hi)
            fm = f(mid)
            if abs(fm) < 1e-6:
                return float(mid)
            if flo * fm <= 0:
                hi = mid
                fhi = fm
            else:
                lo = mid
                flo = fm
        return float(0.5 * (lo + hi))

@dataclass
class BacktestOut:
    equity: np.ndarray
    pnl: np.ndarray
    trades: int
    rfqs: int
    fill_rate: float
    net_pnl_pct: float
    max_dd_pct: float
    es_tick_099_pct: float
    es_day_099_pct: float
    final_inventory: float

def run_backtest(
    steps: int = 3000,
    scenario: str = "normal",
    seed: int = 42,
    ticks_per_day: int = 1000,
    inv_scale: float = 250.0
) -> BacktestOut:

    rng = np.random.default_rng(seed)
    market = SyntheticMarket(MarketParams(), rng)
    rfqp = RFQParams()
    qpar = QCSRFParams()
    model = QCSRF(qpar, rng)
    st = QCSRFState()

    equity0 = 1_000_000.0
    eq = equity0
    equity = np.zeros(steps, dtype=float)
    pnl = np.zeros(steps, dtype=float)

    hour = 8
    state_hist: List[np.ndarray] = []
    sign_hist: List[int] = []

    trades = 0
    rfq_count = 0
    Eref = 0.55

    for t in range(steps):
        if t % 400 == 0:
            hour = (hour + 1) % 24

        mk = market.step(scenario, t)

        x = np.array([safe_log(mk.mid), mk.spread, mk.imbalance, safe_log(mk.depth_exec)], dtype=float)
        state_hist.append(x)
        if len(state_hist) > 1200:
            state_hist = state_hist[-1200:]
        Hx = np.vstack(state_hist)
        trRic, froRic = ricci_proxy(Hx)

        mids = np.array(market.mid_hist, dtype=float)
        rets = np.diff(np.log(np.maximum(mids, EPS)))
        repsi = quantum_repsi(rets, rng)
        E = hurst_rs(mids[-800:])

        rfqs = sample_rfqs(rng, rfqp, hour, mk.vol_1m)
        rfq_count += len(rfqs)

        step_pnl = 0.0

        for F in rfqs:
            sign_hist.append(F.eps)
            if len(sign_hist) > 200:
                sign_hist = sign_hist[-200:]
            H = shannon_entropy_signs(np.array(sign_hist, dtype=int))

            S = abs(st.I) / max(qpar.I_safe, EPS)
            X = np.array([mk.vol_1m / 0.01, safe_log(mk.depth_exec) / 20.0, float(F.tier - 1), float(F.hour) / 23.0], dtype=float)

            lam_slip = model.lambda_slip(froRic)
            cslip_bp = model.c_slip_bp(F.v, mk.depth_exec, lam_slip)

            s_star = model.solve_s_star(F.v, X, E, Eref, H, S, repsi, cslip_bp)
            p_hit = model.p_hit(s_star, X, E, Eref, H, S, repsi)

            if float(rng.random()) < p_hit:
                trades += 1
                step_pnl += (s_star * 1e-4) * F.v
                step_pnl -= (cslip_bp * 1e-4) * F.v

                st.I += F.eps * F.v
                st.I = clip(st.I, -3.0 * qpar.I_max, 3.0 * qpar.I_max)

                lam_inv = model.lambda_inv(E, Eref, H, trRic)
                cinv = model.cost_inv(st.I, F.eps, F.v, lam_inv)
                step_pnl -= cinv * inv_scale

                if abs(st.I) > qpar.I_max:
                    step_pnl -= qpar.forced_hedge_k * abs(st.I)
                    st.I = 0.0

            dH = (H - st.last_H)
            k_skew = qpar.k0 * safe_exp(-qpar.gamma_entropy * dH, lo=-10, hi=10)

            z = abs(st.I) / max(qpar.I_safe, EPS)
            z = min(z, 1e4)
            st.k_hedge += qpar.hedge_eps * (z * z - 1.0) * (1.0 if st.I >= 0 else -1.0)
            st.k_hedge = clip(st.k_hedge, -1e6, 1e6)

            u = F.eps * F.v * p_hit
            h = max(0.0, abs(st.I) - qpar.I_safe) * (1.0 if st.I >= 0 else -1.0)
            st.I += (-k_skew * (st.I + u) - st.k_hedge * h) * 0.01
            st.I = clip(st.I, -3.0 * qpar.I_max, 3.0 * qpar.I_max)

            st.last_H = H

        eq += step_pnl
        if not np.isfinite(eq):
            eq = equity0

        pnl[t] = step_pnl
        equity[t] = eq

    fill_rate = trades / max(rfq_count, 1)
    net_pnl_pct = (equity[-1] - equity0) / equity0 * 100.0
    max_dd_pct = max_drawdown_pct(equity)

    pnl_tick_pct = (pnl / equity0) * 100.0
    es_tick_099 = es_left_tail(pnl_tick_pct, 0.99)

    b = max(int(ticks_per_day), 1)
    n = (len(pnl) // b) * b
    if n >= b:
        pnl_day = pnl[:n].reshape(-1, b).sum(axis=1)
        pnl_day_pct = (pnl_day / equity0) * 100.0
        es_day_099 = es_left_tail(pnl_day_pct, 0.99)
    else:
        es_day_099 = float("nan")

    return BacktestOut(
        equity=equity,
        pnl=pnl,
        trades=trades,
        rfqs=rfq_count,
        fill_rate=float(fill_rate),
        net_pnl_pct=float(net_pnl_pct),
        max_dd_pct=float(max_dd_pct),
        es_tick_099_pct=float(es_tick_099),
        es_day_099_pct=float(es_day_099),
        final_inventory=float(st.I),
    )

if __name__ == "__main__":
    out = run_backtest(steps=3000, scenario="normal", seed=42, ticks_per_day=1000, inv_scale=250.0)
    print("=== Scenario: normal ===")
    print("Net PnL %          :", round(out.net_pnl_pct, 4))
    print("Max Drawdown %     :", round(out.max_dd_pct, 4))
    print("ES/CVaR0.99 tick % :", round(out.es_tick_099_pct, 6))
    print("ES/CVaR0.99 day  % :", round(out.es_day_099_pct, 6))
    print("Trades             :", out.trades)
    print("RFQs               :", out.rfqs)
    print("Fill rate          :", round(out.fill_rate, 4))
    print("Final Inventory    :", round(out.final_inventory, 2))


=== Scenario: normal ===
Net PnL %          : 26.113
Max Drawdown %     : 0.4234
ES/CVaR0.99 tick % : 0.000471
ES/CVaR0.99 day  % : 0.0
Trades             : 289
RFQs               : 1231
Fill rate          : 0.2348
Final Inventory    : -6437365.34
